# Day 29 — Model serving & cost at scale

Whichever door you picked (Day 28), you now have to size and pay for the serving layer. This
hour: the throughput fundamentals (prefill vs decode, batching, KV cache), how many replicas a
latency SLA needs, the cost levers ranked by savings, the self-host-vs-API break-even, and a
runnable calculator for all of it.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | Where the time and money go | 4 min |
| 1 | Throughput: prefill, decode, batching, the KV-cache ceiling | 14 min |
| 2 | Capacity: replicas for a p95 SLA | 12 min |
| 3 | Cost levers, ranked | 14 min |
| 4 | Self-host vs API break-even | 8 min |
| 5 | A serving cost calculator | 5 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import numpy as np, math
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

ready


## 0 — Where the time and money go (4 min)

An LLM request is two phases:

1. **Prefill** — process the whole prompt in one parallel pass. Compute-bound. Cost ∝ input
   tokens. Fills the KV cache.
2. **Decode** — generate output tokens one at a time, each a full forward pass reading the KV
   cache. Memory-bandwidth-bound. Cost ∝ output tokens. This is where the wall-clock goes.

So: **input tokens are cheap-ish and fast; output tokens are the expensive, slow part.** Every
cost lever is some version of "send fewer input tokens (caching)" or "generate fewer output
tokens (discipline, smaller model, batching)".

## 1 — Throughput (14 min)

In [2]:
# A rough single-GPU model. Numbers are illustrative (order-of-magnitude for a ~8B model on
# one modern accelerator); real numbers come from load-testing YOUR model+hardware.
GPU = dict(
    prefill_tok_per_s = 40_000,     # parallel prefill throughput
    decode_tok_per_s  = 3_000,      # aggregate decode throughput across the batch
    kv_bytes_per_tok  = 128 * 1024, # KV cache bytes per token (all layers), ~128 KB here
    gpu_mem_gb        = 80,
    weights_gb        = 16,
)

def max_concurrent_requests(avg_context_tokens):
    kv_budget = (GPU["gpu_mem_gb"] - GPU["weights_gb"] - 4) * 1e9    # leave 4GB headroom
    return int(kv_budget / (avg_context_tokens * GPU["kv_bytes_per_tok"]))

def request_latency(in_tok, out_tok, batch_size):
    prefill_s = in_tok / GPU["prefill_tok_per_s"]
    # decode throughput is shared across the batch
    per_req_decode_tok_per_s = GPU["decode_tok_per_s"] / max(1, batch_size)
    decode_s = out_tok / per_req_decode_tok_per_s
    return prefill_s + decode_s, prefill_s, decode_s

for ctx in [1_000, 4_000, 16_000, 64_000]:
    print(f"ctx {ctx:>6} tok -> up to {max_concurrent_requests(ctx):>4} concurrent requests "
          f"before the KV cache is full")
print()
for bs in [1, 8, 32, 64]:
    tot, pf, dc = request_latency(2000, 300, bs)
    print(f"batch {bs:>3}: latency {tot:5.2f}s (prefill {pf:.2f} + decode {dc:.2f})  "
          f"throughput ~{bs/tot:.1f} req/s")

ctx   1000 tok -> up to  457 concurrent requests before the KV cache is full
ctx   4000 tok -> up to  114 concurrent requests before the KV cache is full
ctx  16000 tok -> up to   28 concurrent requests before the KV cache is full
ctx  64000 tok -> up to    7 concurrent requests before the KV cache is full

batch   1: latency  0.15s (prefill 0.05 + decode 0.10)  throughput ~6.7 req/s
batch   8: latency  0.85s (prefill 0.05 + decode 0.80)  throughput ~9.4 req/s
batch  32: latency  3.25s (prefill 0.05 + decode 3.20)  throughput ~9.8 req/s
batch  64: latency  6.45s (prefill 0.05 + decode 6.40)  throughput ~9.9 req/s


The two knobs fight each other:

- **Bigger batch → higher total throughput** (the GPU is busy) **but higher per-request
  latency** (decode bandwidth is split N ways).
- **Longer context → fewer concurrent requests** (KV cache fills up) regardless of batch.

**Continuous batching** (vLLM, TGI, TensorRT-LLM) is the standard fix: instead of waiting for a
whole batch to finish, it adds and removes requests from the running batch every step, so a
short request doesn't wait behind a long one. It roughly 2–4×'s real-world throughput over
naive static batching.

In [3]:
# continuous vs static batching, illustratively
def static_batch_throughput(reqs_out_tokens, batch=32):
    # whole batch runs until the LONGEST request finishes -> short reqs waste slots
    groups = [reqs_out_tokens[i:i+batch] for i in range(0, len(reqs_out_tokens), batch)]
    total_tok = sum(max(g) * len(g) for g in groups)      # padded to the max
    return sum(reqs_out_tokens) / total_tok

lengths = rng.integers(20, 600, size=512)      # varied output lengths
util = static_batch_throughput(lengths)
print(f"static batching GPU utilisation ~{util:.0%} (rest is padding waste)")
print(f"continuous batching recovers most of that -> ~2-4x effective throughput")

static batching GPU utilisation ~55% (rest is padding waste)
continuous batching recovers most of that -> ~2-4x effective throughput


## 2 — Capacity: replicas for a p95 SLA (12 min)

Given request rate `λ` and per-request service time, how many replicas keep p95 latency under
your target? Model each replica as an M/M/1-ish queue; a request waits when its replica is
busy. Use the Erlang-C approximation for `c` servers.

In [4]:
def erlang_c(lam, mu, c):
    a = lam / mu                       # offered load in Erlangs
    if a >= c: return 1.0              # unstable — queue grows without bound
    s = sum(a**k / math.factorial(k) for k in range(c))
    last = a**c / (math.factorial(c) * (1 - a/c))
    return last / (s + last)           # P(wait) — probability an arrival must queue

def p95_latency(lam, service_s, c):
    mu = 1 / service_s
    pw = erlang_c(lam, mu, c)
    # mean wait in queue, then p95 ~ mean_wait_tail + service tail (rough)
    wq = pw / (c * mu - lam) if c * mu > lam else 1e9
    # crude p95: 3x mean queue wait + 1.5x service time
    return 3 * wq + 1.5 * service_s

RPS = 8
SERVICE_S = 1.6          # from §1: a typical request
TARGET_P95 = 4.0
print(f"offered load = {RPS*SERVICE_S:.1f} Erlangs -> need c > {math.ceil(RPS*SERVICE_S)} just for stability\n")
met = None
for c in range(1, 30):
    p95 = p95_latency(RPS, SERVICE_S, c)
    util = RPS * SERVICE_S / c
    ok = p95 <= TARGET_P95 and util < 0.80
    if met is None and ok: met = c
    if c <= max(20, (met or 0) + 2):
        print(f"c={c:2d}  util={util:4.0%}  p95~{min(p95,99):5.2f}s{'  <- meets SLA' if ok else ''}")

offered load = 12.8 Erlangs -> need c > 13 just for stability

c= 1  util=1280%  p95~99.00s
c= 2  util=640%  p95~99.00s
c= 3  util=427%  p95~99.00s
c= 4  util=320%  p95~99.00s
c= 5  util=256%  p95~99.00s
c= 6  util=213%  p95~99.00s
c= 7  util=183%  p95~99.00s
c= 8  util=160%  p95~99.00s
c= 9  util=142%  p95~99.00s
c=10  util=128%  p95~99.00s
c=11  util=116%  p95~99.00s
c=12  util=107%  p95~99.00s
c=13  util= 98%  p95~24.87s
c=14  util= 91%  p95~ 5.05s
c=15  util= 85%  p95~ 3.40s
c=16  util= 80%  p95~ 2.86s
c=17  util= 75%  p95~ 2.63s  <- meets SLA
c=18  util= 71%  p95~ 2.51s  <- meets SLA
c=19  util= 67%  p95~ 2.46s  <- meets SLA
c=20  util= 64%  p95~ 2.43s  <- meets SLA


Two rules the numbers show:

1. **Never run a replica above ~70–80% utilisation.** The queue-wait term blows up as
   utilisation → 1 (that's the `1/(cμ−λ)` denominator). The last 20% of "capacity" costs you
   the tail.
2. **Headroom is not waste — it's the SLA.** Size for peak-ish load at 70% util, and let
   autoscaling handle the rest.

### Autoscaling

- Scale on a **leading signal** (queue depth, concurrent requests, or tokens/sec) not just
  CPU/GPU util — util is a lagging, noisy signal for LLM serving.
- Account for **scale-up lag**: a new GPU replica can take **minutes** (pull image, load
  weights). Pre-warm, keep a floor of replicas, or use provisioned concurrency.
- Scale down slowly (long cooldown) to avoid thrash.

## 3 — Cost levers, ranked (14 min)

Ranked by typical savings-per-effort. Apply top-down; measure after each.

In [5]:
BASE = dict(calls_per_month=5_000_000, in_tok=1200, out_tok=250, pin=3/1e6, pout=15/1e6)

def monthly_cost(calls_per_month, in_tok, out_tok, pin, pout):
    return calls_per_month * (in_tok * pin + out_tok * pout)

base_cost = monthly_cost(**BASE)
print(f"baseline: ${base_cost:,.0f}/month\n")

levers = []
# 1. prompt caching: 70% of input is a stable prefix, cached at ~10% cost
cached = monthly_cost(BASE["calls_per_month"], BASE["in_tok"]*(0.3 + 0.7*0.1), BASE["out_tok"], BASE["pin"], BASE["pout"])
levers.append(("prompt caching (70% stable prefix)", cached))
# 2. semantic cache: 25% of queries are near-duplicates -> served free
sem = base_cost * 0.75
levers.append(("semantic cache (25% hit rate)", sem))
# 3. output discipline: max_tokens + stop seqs cut mean output 250 -> 140
disc = monthly_cost(BASE["calls_per_month"], BASE["in_tok"], 140, BASE["pin"], BASE["pout"])
levers.append(("output-token discipline (250->140)", disc))
# 4. model right-size / cascade: 70% handled by a model at 1/5 the price
casc = 0.3*base_cost + 0.7*monthly_cost(BASE["calls_per_month"], BASE["in_tok"], BASE["out_tok"], BASE["pin"]/5, BASE["pout"]/5)
levers.append(("cascade: 70% to a 5x-cheaper model", casc))
# 5. batch 40% of traffic (async) at 50% off
bat = 0.6*base_cost + 0.4*base_cost*0.5
levers.append(("move 40% of traffic to batch (-50%)", bat))

print(f"{'lever':40s} {'$/month':>12} {'saving':>10}")
for name, cost in sorted(levers, key=lambda x: x[1]):
    print(f"{name:40s} {cost:>12,.0f} {1-cost/base_cost:>9.0%}")
print("\ncombined (multiplicative on the parts they touch) can be 60-85% off the baseline.")

baseline: $36,750/month

lever                                         $/month     saving
cascade: 70% to a 5x-cheaper model             16,170       56%
prompt caching (70% stable prefix)             25,410       31%
semantic cache (25% hit rate)                  27,562       25%
output-token discipline (250->140)             28,500       22%
move 40% of traffic to batch (-50%)            29,400       20%

combined (multiplicative on the parts they touch) can be 60-85% off the baseline.


### The lever list (Anthropic-style ordering: free wins first)

| # | Lever | Typical saving | Effort |
| - | ----- | -------------- | ------ |
| 1 | **Prompt caching** — put the stable prefix first, cache it | 30–90% of input cost | trivial |
| 2 | **Input hygiene** — trim retrieved chunks, drop dead context, compact history | 10–40% of input | low |
| 3 | **Output discipline** — right `max_tokens`, `stop_sequences`, "be concise" | 20–50% of output | low |
| 4 | **Semantic cache** — embed the query, serve a cached answer on a near-hit | = hit rate | medium |
| 5 | **Model right-sizing / cascade** (Day 11) — cheapest model that clears the bar; escalate on doubt | 40–80% | medium |
| 6 | **Batch API** — anything not needing a synchronous answer | ~50% on that slice | low |
| 7 | **Lower `effort` / smaller thinking budget** on routes that don't need it | 10–40% | low |
| 8 | (self-host) **quantization** (int8/4-bit), **spot** instances, **continuous batching** | 30–70% of infra | high |

Rule: **free wins before tradeoffs.** Caching and token hygiene don't touch quality; model
downgrades and effort cuts do — measure those against your eval (Day 26).

## 4 — Self-host vs API break-even (8 min)

In [6]:
API_PRICE_PER_1M_BLENDED = 8.0        # blended $/1M tokens for a mid model
GPU_HOURLY = 3.5                       # one accelerator, on-demand cloud
GPU_TOK_PER_S_SUSTAINED = 2200         # realistic sustained throughput (prefill+decode mixed)
UTIL = 0.6                             # you can't keep it 100% busy

def api_cost(tokens_per_month):
    return tokens_per_month / 1e6 * API_PRICE_PER_1M_BLENDED

def selfhost_cost(tokens_per_month):
    tok_per_gpu_month = GPU_TOK_PER_S_SUSTAINED * UTIL * 3600 * 730
    gpus = max(1, math.ceil(tokens_per_month / tok_per_gpu_month))
    return gpus * GPU_HOURLY * 730, gpus

vols = np.array([1e7, 1e8, 5e8, 2e9, 1e10, 5e10])
print(f"{'tokens/month':>14} {'API $':>12} {'self-host $':>12} {'gpus':>5}  cheaper")
for v in vols:
    a = api_cost(v); s, g = selfhost_cost(v)
    print(f"{v:>14,.0f} {a:>12,.0f} {s:>12,.0f} {g:>5}  {'self-host' if s < a else 'API'}")

  tokens/month        API $  self-host $  gpus  cheaper
    10,000,000           80        2,555     1  API
   100,000,000          800        2,555     1  API
   500,000,000        4,000        2,555     1  self-host
 2,000,000,000       16,000        2,555     1  self-host
10,000,000,000       80,000        7,665     3  self-host
50,000,000,000      400,000       38,325    15  self-host


Self-hosting only wins at **sustained high volume**, and the sticker price (GPU hours) is the
easy part. The real costs: an ML-serving/on-call team, load-testing, autoscaling, model
updates, security patching, evals for your fine-tune, and the opportunity cost of not shipping
features. For most teams the API (or Bedrock) is cheaper *all-in* well past the raw-token
break-even. Self-host when you have hard data-residency needs, a heavily-used fine-tune, or
genuinely enormous steady volume.

## 5 — A serving cost calculator (5 min)

In [7]:
def serving_report(*, rps, in_tok, out_tok, pin, pout, target_p95=4.0,
                   cache_prefix_frac=0.0, semantic_hit=0.0, out_discipline=1.0):
    calls_mo = rps * 3600 * 730
    eff_in = in_tok * (1 - cache_prefix_frac + cache_prefix_frac * 0.1)
    eff_out = out_tok * out_discipline
    gross = monthly_cost(calls_mo, eff_in, eff_out, pin, pout)
    net = gross * (1 - semantic_hit)
    service_s, *_ = request_latency(in_tok, int(out_tok*out_discipline), batch_size=16)
    c = next((c for c in range(1, 200)
              if p95_latency(rps*(1-semantic_hit), service_s, c) <= target_p95
              and rps*(1-semantic_hit)*service_s/c < 0.8), 200)
    return dict(calls_per_month=int(calls_mo), effective_in_tok=round(eff_in),
                gross_month=round(gross), net_month=round(net),
                service_s=round(service_s, 2), replicas_for_sla=c)

import json
print("baseline:")
print(json.dumps(serving_report(rps=20, in_tok=1200, out_tok=250, pin=3/1e6, pout=15/1e6), indent=1))
print("\nwith caching + semantic cache + output discipline:")
print(json.dumps(serving_report(rps=20, in_tok=1200, out_tok=250, pin=3/1e6, pout=15/1e6,
        cache_prefix_frac=0.7, semantic_hit=0.25, out_discipline=0.6), indent=1))

baseline:
{
 "calls_per_month": 52560000,
 "effective_in_tok": 1200,
 "gross_month": 386316,
 "net_month": 386316,
 "service_s": 1.36,
 "replicas_for_sla": 35
}

with caching + semantic cache + output discipline:
{
 "calls_per_month": 52560000,
 "effective_in_tok": 444,
 "gross_month": 188270,
 "net_month": 141202,
 "service_s": 0.83,
 "replicas_for_sla": 16
}


## 6 — Exercises

1. **Batch-size / latency curve.** Plot per-request latency and aggregate throughput
   (req/s) vs batch size 1–128 for a 2000-in/300-out request. Mark the batch size where
   latency crosses a 3s SLA. What throughput do you get there?
2. **KV-cache wall.** For contexts of 2k, 8k, 32k, 128k tokens, compute max concurrent
   requests. At 128k, how many GPUs to serve 200 concurrent long-context sessions?
3. **Utilisation cliff.** Fix `c` and sweep RPS so utilisation goes 0.5 → 0.95. Plot p95
   latency. Where does it become unacceptable, and what utilisation is that?
4. **Lever stacking.** Apply levers 1, 3, 4, 5 from §3 in sequence to the `BASE` workload,
   recomputing cost after each. Plot the running cost. Which single lever moved it most?
5. **Break-even with team cost.** Add a `$400k/year` platform-team cost to `selfhost_cost`.
   Re-run the table. How much does the token-volume break-even shift?
6. **Cache staleness.** A semantic cache with TTL `T` and query rate `λ`: model the fraction
   of hits that serve a *stale* answer if the underlying data changes every `D` days. Pick a
   TTL that keeps stale-hit rate < 1%.

> **Attempt every exercise and the quiz first.** The worked solutions and the answer key live in [`solutions/solutions.ipynb`](solutions/solutions.ipynb) — open it only to check your work, not to start.

## Self-check quiz


1. What are the two phases of an LLM request, and which one dominates wall-clock time?
2. Why does a bigger serving batch raise per-request latency?
3. What limits the number of concurrent long-context requests on one GPU?
4. What does continuous batching fix, versus static batching?
5. Why should you never run a serving replica above ~80% utilisation?
6. List the cost levers in order of "apply first", and which ones are free (don't touch
   quality).
7. When does self-hosting actually beat the API, all-in?

## Where this goes next

- **Day 30 — Deploy the RAG pipeline:** package Week 6's pipeline as an API endpoint —
  Lambda handler, API Gateway, cold-start mitigation, auth, and the IaC — with a local harness
  that invokes the handler like Lambda would.